<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/c10_linear_regression_tensors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import torch
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

dataset = fetch_california_housing()  # only numerical, no missing values
X = dataset.data
y = dataset.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

print(f"shape X: {X_train.shape, X_val.shape, X_test.shape}")
print(f"shape y: {y_train.shape, y_val.shape, y_test.shape}")


shape X: ((16512, 8), (2064, 8), (2064, 8))
shape y: ((16512,), (2064,), (2064,))


In [24]:
X_train = torch.FloatTensor(X_train) #convert to tensors
X_val = torch.FloatTensor(X_val)
X_test = torch.FloatTensor(X_test)

# Normalize, we could use StandardScaler but let's do it manually
means = X_train.mean(dim=0, keepdim=True)
stds = X_train.std(dim=0, keepdim=True)
print(f'shape means: {means.shape}')




shape means: torch.Size([1, 8])


In [25]:
X_train = (X_train - means) / stds
X_val = (X_val - means) / stds
X_test = (X_test - means) / stds

y_train = torch.FloatTensor(y_train).reshape(-1,1)
y_val = torch.FloatTensor(y_val).reshape(-1,1)
y_test = torch.FloatTensor(y_test).reshape(-1,1)

print(f'shape X_train: {X_train.shape}')

shape X_train: torch.Size([16512, 8])


In [26]:
torch.manual_seed(42)
n_features = X_train.shape[1] #X_train.shape = (16512, 8), here we take 8
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)  # real number for the moment, but we will use broadcasting!
print(b.shape)

torch.Size([])


In [27]:
learning_rate = 0.4
n_epochs = 20

for epoch in range(n_epochs):
    y_pred = X_train @ w + b #prediction vector writing one prediction y=x_1*theta_1+...+x_n*theta_n+b
    loss = ((y_pred-y_train)**2).mean()
    loss.backward()  # compute the gradients of the loss functions with respect to every parameter
    with torch.no_grad():
        w -= learning_rate * w.grad #these are the "usual" gradients wrt w_1,....w_n
        b -= learning_rate * b.grad # this is the gradient of b (w_0)
        w.grad.zero_()
        b.grad.zero_()
    print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item()}")

Epoch 1/20, Loss: 16.044116973876953
Epoch 2/20, Loss: 4.6996378898620605
Epoch 3/20, Loss: 2.1376359462738037
Epoch 4/20, Loss: 1.2627451419830322
Epoch 5/20, Loss: 0.9295777678489685
Epoch 6/20, Loss: 0.7914218306541443
Epoch 7/20, Loss: 0.726655125617981
Epoch 8/20, Loss: 0.6907863616943359
Epoch 9/20, Loss: 0.6671205759048462
Epoch 10/20, Loss: 0.6492290496826172
Epoch 11/20, Loss: 0.6345410943031311
Epoch 12/20, Loss: 0.6219595074653625
Epoch 13/20, Loss: 0.6109610795974731
Epoch 14/20, Loss: 0.6012548208236694
Epoch 15/20, Loss: 0.5926492214202881
Epoch 16/20, Loss: 0.5850005745887756
Epoch 17/20, Loss: 0.578191876411438
Epoch 18/20, Loss: 0.572124183177948
Epoch 19/20, Loss: 0.5667113661766052
Epoch 20/20, Loss: 0.561878502368927


In [28]:
# let's make predictions on new data
X_new = X_test[:3] #fake new instances, already a tensor
with torch.no_grad():
    y_pred = X_new @ w + b
print(y_pred)

tensor([[2.0760],
        [2.8031],
        [3.1190]])


In [29]:
import torch.nn as nn

torch.manual_seed(42)
model = nn.Linear(in_features=n_features, out_features=1)
print(model.weight, model.bias) # 8 weights (8 features), 1 bias term only one neuron

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True) Parameter containing:
tensor([0.3117], requires_grad=True)


The parameters are random now!

In [30]:
list(model.parameters())

[Parameter containing:
 tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
        requires_grad=True),
 Parameter containing:
 tensor([0.3117], requires_grad=True)]

In [31]:
model(X_train[:3]) #the model is not yet trained! but we can use predict

tensor([[ 0.7173],
        [ 1.0659],
        [-0.2944]], grad_fn=<AddmmBackward0>)

In [32]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item()}")

train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/20, Loss: 4.2877516746521
Epoch 2/20, Loss: 0.771273672580719
Epoch 3/20, Loss: 0.6180822253227234
Epoch 4/20, Loss: 0.5989888310432434
Epoch 5/20, Loss: 0.5885871052742004
Epoch 6/20, Loss: 0.5802777409553528
Epoch 7/20, Loss: 0.5731872916221619
Epoch 8/20, Loss: 0.5670098662376404
Epoch 9/20, Loss: 0.5615824460983276
Epoch 10/20, Loss: 0.5567958950996399
Epoch 11/20, Loss: 0.5525659918785095
Epoch 12/20, Loss: 0.5488236546516418
Epoch 13/20, Loss: 0.5455095171928406
Epoch 14/20, Loss: 0.5425723791122437
Epoch 15/20, Loss: 0.5399671196937561
Epoch 16/20, Loss: 0.5376547574996948
Epoch 17/20, Loss: 0.5356006622314453
Epoch 18/20, Loss: 0.5337745547294617
Epoch 19/20, Loss: 0.5321498513221741
Epoch 20/20, Loss: 0.5307032465934753


In [33]:
X_new = X_test[:3]
with torch.no_grad():
  y_pred = model(X_new)

y_pred

tensor([[2.1266],
        [2.8399],
        [3.1834]])